<a href="https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/feat%2Fm1-t02-colab-resume/notebooks/m1_colab_fast_production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quy trình thu thập dữ liệu chính thức giai đoạn 1 trên Google Colab

**Quy định vận hành:**
- **Môi trường:** Google Colab CPU-only (Runtime → Change runtime type → CPU / accelerator None). Tuyệt đối không dùng GPU/TPU.
- **Đơn tiến trình (Single-writer):** Mỗi corpus chỉ được ghi bởi một tiến trình duy nhất. Không chạy song song nhiều sổ tay cùng ghi vào một thư mục.
- **Bảo mật:** Khóa truy cập Kaggle (`KAGGLE_API_TOKEN`) và Semantic Scholar (`S2_API_KEY`) được nạp an toàn từ Google Colab Secrets (biểu tượng ổ khóa ở thanh bên trái), không viết trực tiếp vào mã nguồn.
- **Kiến trúc tăng tốc:**
  1. Thư mục làm việc cục bộ (`/content/fast_crawling_workspace`): Giảm thiểu độ trễ I/O của Google Drive FUSE.
  2. RAM Disk (`/dev/shm`): Giải nén và lọc tệp TeX/Bib trực tiếp trong bộ nhớ RAM.
  3. Nạp trước siêu dữ liệu Kaggle: Trích xuất lịch sử phiên bản và ngày nộp từ tệp snapshot cục bộ, tiết kiệm hàng nghìn truy vấn discovery.
  4. Sao lưu và khôi phục bền vững: Tự động nén tệp `.tar.gz` kèm mã băm `.sha256` đồng bộ lên Google Drive sau mỗi lô 100 bài báo.

In [ ]:
# Bước 1: Thiết lập môi trường, kết nối Google Drive và kiểm tra mã nguồn
CODE_COMMIT = "489a82fd959ccc6cced7c59fc309255e4421bef2"  # Mốc lịch sử đối chiếu (Historical Pin)
EXECUTION_COMMIT = "69b4b5e758f74d255ff244c56271e294da57b8e4"  # Mã commit thực thi (Runtime Pin)

import os
from pathlib import Path
import re
import subprocess
import sys

if os.environ.get("COLAB_GPU") == "1":
    raise RuntimeError("Chuyển accelerator về None/CPU trong Runtime trước khi chạy.")

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = Path('/content/Arxiv-Project-Group10')
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if not isinstance(CODE_COMMIT, str) or not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise RuntimeError("Bắt buộc phải điền đầy đủ mã commit 40 ký tự cho CODE_COMMIT để bảo đảm tính tái lập (fail-closed).")
if not isinstance(EXECUTION_COMMIT, str) or not re.fullmatch(r"[0-9a-f]{40}", EXECUTION_COMMIT):
    raise RuntimeError("Bắt buộc phải điền đầy đủ mã commit 40 ký tự cho EXECUTION_COMMIT để bảo đảm tính tái lập (fail-closed).")

subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "--detach", EXECUTION_COMMIT], check=True)
current_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if current_head != EXECUTION_COMMIT:
    raise RuntimeError(f"Commit sau khi checkout ({current_head}) không khớp với EXECUTION_COMMIT ({EXECUTION_COMMIT}).")

print(f"Mốc lịch sử đối chiếu (HISTORICAL_CODE_COMMIT): {CODE_COMMIT}")
print(f"Mã nguồn thực thi chính thức (RUNTIME_EXECUTION_COMMIT): {current_head}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Hoàn tất chuẩn bị môi trường và thư viện phụ thuộc.")


In [ ]:
# Bước 2: Nạp cấu hình Kaggle API an toàn từ Colab Secrets và chuẩn bị snapshot
import os
from pathlib import Path

kaggle_token = os.environ.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    try:
        from google.colab import userdata
        kaggle_token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        pass

if kaggle_token:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    (kaggle_dir / "access_token").write_text(kaggle_token.strip() + "\n")
    (kaggle_dir / "access_token").chmod(0o600)
    print("Đã cấu hình Kaggle access token an toàn từ Colab Secrets.")
else:
    print("Chưa cấu hình KAGGLE_API_TOKEN trong Colab Secrets. Bỏ qua tải tự động từ Kaggle.")

snapshot_file = Path("arxiv-metadata-oai-snapshot.json")
if not snapshot_file.exists() and kaggle_token:
    print("Đang tải bộ dữ liệu Cornell-University/arxiv từ Kaggle...")
    !kaggle datasets download -d Cornell-University/arxiv
    !unzip -q -o arxiv.zip
    print("Đã giải nén arxiv-metadata-oai-snapshot.json.")
elif snapshot_file.exists():
    print("Tệp arxiv-metadata-oai-snapshot.json đã sẵn sàng trên ổ đĩa.")

In [ ]:
# Bước 3: Thực thi thu thập mã nguồn TeX và lịch sử phiên bản (Lô 100 định danh)
# Phân bổ 200 lô dữ liệu (20.000 IDs) cho 4 thành viên nhóm (mỗi người 50 lô / 5.000 IDs):
# - MSSV 24120010 (Trần Văn Tấn Khôi): Lô 1..50    (2508.07872 .. 2508.12871)
# - MSSV 24120035 (Lê Thành Đạt):       Lô 51..100  (2507.16659 .. 2507.21658)
# - MSSV 24120232 (Nguyễn Văn Toàn):     Lô 101..150 (2502.08235 .. 2502.13234)
# - MSSV 24120260 (Trương Tuấn Anh):     Lô 151..200 (2501.02642 .. 2501.07641)
import sys, os
from pathlib import Path

REPO_DIR = Path('/content/Arxiv-Project-Group10')
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

from src.fast_production import FastProductionRunner, member_cohort_range, verify_cohort_handoff

# CẤU HÌNH THÀNH VIÊN VÀ CHẾ ĐỘ CHẠY:
MSSV = "24120260"  # Điền mã sinh viên: "24120010", "24120035", "24120232", hoặc "24120260"
MODE = "MEMBER_PRODUCTION"  # "MEMBER_PRODUCTION" (50 lô được giao) hoặc "PILOT" (thử nghiệm 1 lô)

assigned_cohorts = member_cohort_range(MSSV)
if MODE == "PILOT":
    COHORT_LIST = [assigned_cohorts[0]]
    print(f"Chế độ PILOT: Chạy lô thử nghiệm {COHORT_LIST[0]} cho MSSV {MSSV}.")
elif MODE == "MEMBER_PRODUCTION":
    COHORT_LIST = assigned_cohorts
    print(f"Chế độ MEMBER_PRODUCTION: Chạy 50 lô ({COHORT_LIST[0]}..{COHORT_LIST[-1]}) cho MSSV {MSSV}.")
else:
    raise ValueError(f"MODE '{MODE}' không hợp lệ. Chỉ chấp nhận 'PILOT' hoặc 'MEMBER_PRODUCTION'.")

LOCAL_ROOT = '/content/fast_crawling_workspace'
DRIVE_ROOT = '/content/drive/MyDrive/Group10_M1_Production'

runner = FastProductionRunner(
    local_root=LOCAL_ROOT,
    drive_root=DRIVE_ROOT,
    retry_429=True,
    retry_wait_budget=300.0,
    timeout=30.0,
)

for cohort_num in COHORT_LIST:
    restored = runner.restore_cohort(cohort_num)
    if restored > 0:
        print(f"Đã khôi phục {restored} thư mục bài báo của lô {cohort_num} từ bản sao lưu Google Drive.")

snapshot_path = Path('arxiv-metadata-oai-snapshot.json')
if snapshot_path.exists():
    print(f"Đang nạp trước siêu dữ liệu từ snapshot cho các lô {COHORT_LIST[:3]}... ({len(COHORT_LIST)} lô)")
    res = runner.preload_kaggle_discovery(snapshot_path, COHORT_LIST)
    if not res.get('interrupted'):
        print(f"Nạp thành công {res['found_papers']}/{res['target_papers']} bài báo trong {res.get('elapsed_seconds', 0.0):.2f}s.")

for cohort_num in COHORT_LIST:
    print(f"\nBắt đầu thu thập lô {cohort_num} (100 định danh, revision_history=True)...")
    report = runner.run_cohort(cohort_num, resume=True, revision_history=True)
    stats = report.get('fast_runner', {})
    completed = stats.get('completed_papers', 0)
    elapsed = stats.get('elapsed_seconds', 0.0)
    print(f"Lô {cohort_num}: hoàn thành {completed}/100 bài báo (thời gian phiên: {elapsed:.1f}s).")
    print(f"Bản sao lưu trên Drive: {stats.get('archive_path')}")
    if stats.get('stopped'):
        print(f"Dừng chuỗi xử lý do điều kiện bảo vệ: {stats.get('stop_reason')}")
        break
    if completed == 100:
        try:
            h_info = verify_cohort_handoff(runner, cohort_num)
            print(f"Lô {cohort_num}: ĐÃ ĐẠT CHUẨN BÀN GIAO M1 (archive_state={h_info['archive_state']}).")
        except Exception as e:
            print(f"Lô {cohort_num}: Kiểm toán chưa đạt chuẩn bàn giao: {e}")


In [ ]:
# Bước 4: Làm giàu dữ liệu trích dẫn và siêu dữ liệu Semantic Scholar (S2 Enrichment)
import os

s2_api_key = os.environ.get("SEMANTIC_SCHOLAR_API_KEY") or os.environ.get("S2_API_KEY")
if not s2_api_key:
    try:
        from google.colab import userdata
        s2_api_key = userdata.get("SEMANTIC_SCHOLAR_API_KEY") or userdata.get("S2_API_KEY")
    except Exception:
        pass

if s2_api_key:
    s2_key_clean = s2_api_key.strip()
    os.environ["SEMANTIC_SCHOLAR_API_KEY"] = s2_key_clean
    os.environ["S2_API_KEY"] = s2_key_clean
    print("Đã kích hoạt chế độ có xác thực Semantic Scholar API.")
    s2_pacing = 1.0
else:
    print("Chạy ở chế độ không xác thực Semantic Scholar (giới hạn 100 requests / 5 phút). Pacing: 3.0s.")
    s2_pacing = 3.0

s2_cfg = {
    "timeout": 30.0,
    "max_attempts": 3,
    "min_interval": s2_pacing,
    "reference_limit": 100,
}

for cohort_num in COHORT_LIST:
    print(f"\nBắt đầu làm giàu dữ liệu Semantic Scholar cho lô {cohort_num}...")
    s2_report = runner.enrich_cohort_s2(cohort_num, s2_config=s2_cfg)
    enriched = s2_report.get("enriched_papers", 0)
    skipped = s2_report.get("skipped_incomplete", 0)
    print(f"Lô {cohort_num}: làm giàu thành công {enriched} bài báo ({skipped} bài chưa hoàn tất source được bỏ qua).")
    print(f"Bản sao lưu S2 trên Drive: {s2_report.get('archive_path')}")


In [ ]:
# Bước 5: Kiểm toán chất lượng dữ liệu và tạo bảng báo cáo Milestone 1
import sys, os
from pathlib import Path
REPO_DIR = Path('/content/Arxiv-Project-Group10')
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

from scripts.generate_m1_report import compute_m1_statistics, format_markdown_summary
from src.fast_production import cohort_pairs

all_pairs = []
for c in COHORT_LIST:
    all_pairs.extend(cohort_pairs(c))

stats = compute_m1_statistics(runner.store, all_pairs)
report_md = format_markdown_summary(stats)

report_out = Path(DRIVE_ROOT) / "reports" / f"m1_audit_summary_mssv_{MSSV}_{MODE.lower()}.md"
report_out.parent.mkdir(parents=True, exist_ok=True)
report_out.write_text(report_md, encoding="utf-8")
print(f"Đã xuất bảng báo cáo kiểm toán ra Google Drive: {report_out}\n")
print(report_md)


In [ ]:
# Bước 6: (Tùy chọn) Xuất tập dữ liệu bảng (papers & references) dạng Parquet và CSV
# LƯU Ý: Theo Storage Contract v2, Raw Corpus trên ổ đĩa là nguồn dữ liệu chân lý (Source of Truth) của Milestone 1.
# Bước xuất bảng bên dưới chỉ là tiện ích hỗ trợ, hoàn toàn độc lập và không ảnh hưởng tới tính hợp lệ của việc bàn giao M1.
import sys, os
from pathlib import Path
REPO_DIR = Path('/content/Arxiv-Project-Group10')
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

DRIVE_ROOT = '/content/drive/MyDrive/Group10_M1_Production'
from src.export import export_papers_table, export_references_table, export_to_parquet, validate_exported_tables
import pandas as pd

export_dir = Path(DRIVE_ROOT) / "final_output"
export_dir.mkdir(parents=True, exist_ok=True)

try:
    try:
        papers_rows = export_papers_table(runner.store, all_pairs, verified_only=True)
    except TypeError:
        papers_rows = export_papers_table(runner.store, all_pairs)

    try:
        refs_rows = export_references_table(runner.store, all_pairs, verified_only=True)
    except TypeError:
        refs_rows = export_references_table(runner.store, all_pairs)

    val_res = validate_exported_tables(papers_rows, refs_rows)
    print("Kết quả kiểm tra hợp chuẩn schema xuất bản:", val_res)
    if not val_res["is_valid"]:
        print("Cảnh báo vi phạm schema:", val_res["issues"])

    export_to_parquet(papers_rows, export_dir / "papers.parquet")
    export_to_parquet(refs_rows, export_dir / "references.parquet")
    pd.DataFrame(papers_rows).to_csv(export_dir / "papers.csv", index=False)
    pd.DataFrame(refs_rows).to_csv(export_dir / "references.csv", index=False)
    print(f"Đã xuất thành công {len(papers_rows)} bài báo và {len(refs_rows)} quan hệ trích dẫn ra thư mục: {export_dir}")
except Exception as exc:
    print(f"Lưu ý: Xuất bảng gặp lỗi (không ảnh hưởng tới tính toàn vẹn của Raw Corpus): {exc}")
